In [ ]:
# IMPORTS

import requests
from bs4 import BeautifulSoup
import json
import time
from tqdm import tqdm
import re
from io import BytesIO
from pdfminer.high_level import extract_text
import os



# DIRECTORIES

BASE = "data"

ARXIV_HTML = os.path.join(BASE, "arxiv/html")
ARXIV_PDF = os.path.join(BASE, "arxiv/pdf")
ARXIV_JSON = os.path.join(BASE, "arxiv/json")

PUBMED_HTML = os.path.join(BASE, "pubmed/html")
PUBMED_JSON = os.path.join(BASE, "pubmed/json")

for d in [ARXIV_HTML, ARXIV_PDF, ARXIV_JSON, PUBMED_HTML, PUBMED_JSON]:
    os.makedirs(d, exist_ok=True)


# HELPERS

def safe_filename(text):
    return re.sub(r'[^\w\-_.]', '_', text)


def save_text(content, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)


def save_binary(content, path):
    with open(path, "wb") as f:
        f.write(content)

def load_dois(path):
    dois = []
    with open(path) as f:
        for line in f:
            line = line.strip().split("#")[0].strip()
            if line:
                dois.append(line.replace("https://doi.org/", ""))
    return dois



# ARXIV

def extract_arxiv_id(doi):
    m = re.search(r'arxiv\.?(\d+\.\d+)', doi.lower())
    return m.group(1) if m else None


def get_arxiv_html(arxiv_id, headers):
    for v in ["v1", "v2", "v3", ""]:
        url = f"https://arxiv.org/html/{arxiv_id}{v}"
        try:
            r = requests.get(url, headers=headers, timeout=15)
            if r.status_code == 200 and "ltx_page_main" in r.text:
                return r.text
        except:
            pass
    return None


def get_arxiv_pdf(arxiv_id, headers):
    try:
        r = requests.get(f"https://arxiv.org/pdf/{arxiv_id}.pdf", headers=headers)
        if r.status_code == 200:
            return r.content
    except:
        pass
    return None


def parse_arxiv_html(html):
    soup = BeautifulSoup(html, "html.parser")

    title = soup.title.get_text(strip=True) if soup.title else ""

    abs_div = soup.find(id="abstract") or soup.find("div", class_="ltx_abstract")
    abstract = abs_div.get_text(" ", strip=True) if abs_div else ""

    main = soup.find("div", class_="ltx_page_main")
    sections = []

    if not main:
        return title, abstract, sections

    stack = []

    def get_level(tag):
        if tag.name == "h2":
            return 1
        elif tag.name == "h3":
            return 2
        elif tag.name == "h4":
            return 3
        return 1

    for sec in main.find_all("section", class_=re.compile("ltx_(sub)*section")):

        header = sec.find(["h2", "h3", "h4"])
        if not header:
            continue

        heading = header.get_text(strip=True)

        if any(x in heading.lower() for x in ["reference", "bibliography", "acknowledg"]):
            continue

        level = get_level(header)

        text = " ".join(
            p.get_text(" ", strip=True)
            for p in sec.find_all("p")
        )

        node = {
            "heading": heading,
            "text": text,
            "subsections": []
        }

        while stack and stack[-1]["level"] >= level:
            stack.pop()

        if stack:
            stack[-1]["node"]["subsections"].append(node)
        else:
            sections.append(node)

        stack.append({"level": level, "node": node})

    return title, abstract, sections


# PUBMED

def get_pmc_from_doi(doi):
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        r = requests.get(
            f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term={doi}&retmode=json",
            headers=headers
        )
        pmids = r.json()["esearchresult"]["idlist"]

        if not pmids:
            return None

        pmid = pmids[0]

        r = requests.get(
            f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/elink.fcgi?dbfrom=pubmed&db=pmc&id={pmid}&retmode=json",
            headers=headers
        )

        linksets = r.json()["linksets"]
        if not linksets or not linksets[0].get("linksetdbs"):
            return None

        pmcid = linksets[0]["linksetdbs"][0]["links"][0]
        return f"https://www.ncbi.nlm.nih.gov/pmc/articles/PMC{pmcid}/"

    except:
        return None


def get_pmc_html(url, headers):
    try:
        r = requests.get(url, headers=headers)
        if r.status_code == 200:
            return r.text
    except:
        pass
    return None


def parse_pmc_html(html):
    soup = BeautifulSoup(html, "html.parser")

    title = soup.find("h1").get_text(strip=True) if soup.find("h1") else ""

    abs_div = soup.find("div", class_="abstract")
    abstract = abs_div.get_text(" ", strip=True) if abs_div else ""

    sections = []
    stack = []

    def get_level(tag):
        if tag.name == "h2":
            return 1
        elif tag.name == "h3":
            return 2
        elif tag.name == "h4":
            return 3
        return 1

    for sec in soup.find_all("section"):
        header = sec.find(["h2", "h3", "h4"])
        if not header:
            continue

        heading = header.get_text(strip=True)
        level = get_level(header)

        text = " ".join(
            p.get_text(" ", strip=True)
            for p in sec.find_all("p")
        )

        node = {
            "heading": heading,
            "text": text,
            "subsections": []
        }

        while stack and stack[-1]["level"] >= level:
            stack.pop()

        if stack:
            stack[-1]["node"]["subsections"].append(node)
        else:
            sections.append(node)

        stack.append({"level": level, "node": node})

    return title, abstract, sections


# PDF PARSER

def parse_pdf(pdf_bytes):
    try:
        text = extract_text(BytesIO(pdf_bytes))
    except:
        return "", "", []

    if not text:
        return "", "", []

    lines = [l.strip() for l in text.split("\n") if l.strip()]

    title = " ".join(lines[:10])

    abstract = ""
    m = re.search(r'abstract\s*(.*?)(\n\s*\n|\n\d+\s+[A-Z])', text, re.I | re.S)
    if m:
        abstract = re.sub(r'\s+', ' ', m.group(1)).strip()

    section_pattern = re.compile(r'^(\d+(\.\d+)*)(\s+)([A-Z].+)')

    sections = []
    stack = []

    def level_of(num):
        return num.count('.') + 1

    for line in lines:

        match = section_pattern.match(line)

        if match:
            num = match.group(1)
            heading = f"{num} {match.group(4)}"
            lvl = level_of(num)

            node = {
                "heading": heading,
                "text": "",
                "subsections": []
            }

            while stack and stack[-1]["level"] >= lvl:
                stack.pop()

            if stack:
                stack[-1]["node"]["subsections"].append(node)
            else:
                sections.append(node)

            stack.append({"level": lvl, "node": node})

        else:
            if stack:
                stack[-1]["node"]["text"] += " " + line

    return title, abstract, sections



# PROCESSING

def process_doi(doi):
    headers = {"User-Agent": "Mozilla/5.0"}

    arxiv_id = extract_arxiv_id(doi)

    # -------- ARXIV --------
    if arxiv_id:
        html = get_arxiv_html(arxiv_id, headers)

        if html:
            save_text(html, os.path.join(ARXIV_HTML, f"{arxiv_id}.html"))
            title, abstract, sections = parse_arxiv_html(html)

        else:
            pdf = get_arxiv_pdf(arxiv_id, headers)
            if not pdf:
                return None

            save_binary(pdf, os.path.join(ARXIV_PDF, f"{arxiv_id}.pdf"))
            title, abstract, sections = parse_pdf(pdf)

        result = {
            "paper_id": doi,
            "source": "arxiv",
            "title": title,
            "abstract": abstract,
            "sections": sections
        }

        save_text(
            json.dumps(result, indent=2, ensure_ascii=False),
            os.path.join(ARXIV_JSON, f"{safe_filename(doi)}.json")
        )

        return result

    # -------- PUBMED --------
    pmc = get_pmc_from_doi(doi)

    if pmc:
        html = get_pmc_html(pmc, headers)
        if not html:
            return None

        save_text(html, os.path.join(PUBMED_HTML, f"{safe_filename(doi)}.html"))

        title, abstract, sections = parse_pmc_html(html)

        result = {
            "paper_id": doi,
            "source": "pubmed",
            "title": title,
            "abstract": abstract,
            "sections": sections
        }

        save_text(
            json.dumps(result, indent=2, ensure_ascii=False),
            os.path.join(PUBMED_JSON, f"{safe_filename(doi)}.json")
        )

        return result

    return None



# MAIN

if __name__ == "__main__":
    dois = load_dois("dois_test.txt")

    results = []

    for doi in tqdm(dois):
        r = process_doi(doi)
        if r:
            results.append(r)
        time.sleep(1)

    print(f"\nCollected {len(results)} articles")

100%|██████████| 9/9 [00:26<00:00,  3.00s/it]


Collected 9 articles
